OBTENCIÓN DE LOS DATOS DE DISTINTAS FUENTES Y LIMPIEZA -> JORGE (RECOPILACIÓN)

# 01 — Data Exploration

## [ENG] Objectives

Explore the raw data sources before enrichment:

- **NASA FIRMS**: VIIRS and MODIS fire detections for Spain (2023, 2024)
- **CLCPlus Backbone**: Land cover classification tiles
- **Open-Meteo**: Historical weather data availability

1. Understand the shape and columns of FIRMS CSVs
2. Compare VIIRS vs MODIS coverage and confidence formats
3. Visualize fire detections on a map of Spain
4. Inspect CLCPlus tile coverage and land cover classes
5. Explore Open-Meteo weather variable ranges

## [ESP] Objetivos

Explorar las fuentes de datos crudos antes del enriquecimiento:

- **NASA FIRMS**: Detecciones de fuego VIIRS y MODIS para España (2023, 2024)
- **CLCPlus Backbone**: Tiles de clasificación de cobertura del suelo
- **Open-Meteo**: Disponibilidad de datos de clima histórico

1. Entender la forma y columnas de los CSVs FIRMS
2. Comparar cobertura VIIRS vs MODIS y formatos de confianza
3. Visualizar detecciones de fuego en un mapa de España
4. Inspeccionar cobertura de tiles CLCPlus y clases de cobertura del suelo
5. Explorar rangos de variables climáticas de Open-Meteo

In [ ]:
# Celda 1 — Recopilación de todos los datos FIRMS y corrección de la confianza
#
# `merge_viirs_modis` hace dos cosas:
#   1. Carga los CSVs crudos de los tres sensores (modis, viirs_snpp, viirs_noaa20)
#      para los años indicados (2023 y 2024) y los concatena en un único DataFrame.
#   2. Llama a `add_unified_confidence` para corregir/unificar la confianza:
#      - MODIS trae confianza NUMÉRICA (0-100) -> se convierte a categórica (l/n/h)
#        usando los rangos de configs/confidence_thresholds.yaml.
#      - VIIRS trae confianza CATEGÓRICA (l/n/h) -> se convierte a numérica (15/50/85).
#      - Los valores originales se conservan en confidence_og_num / confidence_og_cat,
#        y el resultado unificado queda en confidence_cat / confidence_num.
#
# `save_merged` escribe el resultado en data/processed/merged/. Los datos crudos
# de data/raw/ nunca se modifican (principio de inmutabilidad del pipeline).

from wildfire.enrichment.merge_sensors import merge_viirs_modis, save_merged

# 1) Agrupar todos los CSVs FIRMS (VIIRS + MODIS, 2023 + 2024) y corregir la confianza
df_merged = merge_viirs_modis(country="Spain", years=[2023, 2024])

# 2) Guardar el dataset unificado en un CSV de salida (se ejecuta dentro de esta celda)
ruta_salida = save_merged(df_merged, country="Spain")

print(f"Filas unificadas: {len(df_merged):,}")
print(f"Columnas: {df_merged.shape[1]}")
print(f"Distribución de confianza unificada (confidence_cat):")
print(df_merged["confidence_cat"].value_counts().to_string())
print(f"CSV guardado en: {ruta_salida}")

df_merged.head()

Filas unificadas: 47,505
Columnas: 22
Distribución de confianza unificada (confidence_cat):
confidence_cat
n    41852
h     3598
l     2055
CSV guardado en: data\processed\merged\firms_spain_merged.csv
Número de type 0: 13310


In [ ]:
# Celda 2 — Descarga de clima histórico (Open-Meteo) por lotes / Weather download (batched)
#
# [ESP] Esta celda está COMENTADA a propósito: son ~95 lotes de 500 filas
#   en total (~2 h si se hace entero; al reanudar solo se bajan los pendientes,
#   a razón de batch_interval_seconds: 450) y gasta cuota real de la API
#   Open-Meteo. Se ejecuta fuera del notebook, sin supervisión:
#
#       python scripts/enrich_weather_batch.py
#
#   - Reanuda solo desde data/processed/enriched/firms_spain_weather_partial.csv
#     (primera fila sin temperature_2m); la respuesta HTTP cruda queda en
#     data/processed/openmeteo_response.json.
#   - Ante un 429 espera al siguiente límite minuto/hora UTC y reintenta el
#     mismo lote; el límite diario para limpio (exit 3) y al día siguiente se
#     reanuda donde quedó.
#   - Al completarse publica data/processed/enriched/firms_spain_weather.csv
#     (finalize_weather_csv), que es la entrada de la celda 3.
#   - Códigos de salida: 0 completado/max-batches, 3 límite diario,
#     4 esperas consecutivas excesivas, 1 error.
#
# [EN] This cell is intentionally COMMENTED OUT: ~95 batches of 500 rows
#   total (~2 h end-to-end; a resume only fetches the pending batches at
#   450 s interval) that spend real Open-Meteo API quota. Run it
#   outside the notebook, unattended:
#
#       python scripts/enrich_weather_batch.py
#
#   It resumes automatically from firms_spain_weather_partial.csv and, when
#   complete, publishes firms_spain_weather.csv — the input of cell 3.
#   429s are waited out; the daily limit stops cleanly (exit 3).
#
# from wildfire.enrichment.weather_batch import run_weather_enrichment
#
# status = run_weather_enrichment(
#     max_batches=None,        # None = hasta completar / until complete
#     interval_seconds=450,    # openmeteo.batch_interval_seconds
# )
# print(status)


In [ ]:
# Celda 3 — Dataset con clima listo para CLC / Weather dataset ready for CLC
#
# [ESP] load_weather_for_clc aplica el contrato de esta celda:
#   1. Si existe firms_spain_weather.csv (descarga terminada) → se usa tal cual.
#   2. Si no → se parte del parcial y se quedan solo las filas con
#      temperature_2m (lo descargado hasta ahora), para poder avanzar con CLC
#      mientras la descarga sigue en paralelo.
#   3. Si todavía no hay nada de clima → RuntimeError con instrucciones.
#
# [EN] Uses the completed weather CSV if present; otherwise the partial one
#   filtered to the rows that already have temperature_2m.

from wildfire.enrichment.weather_batch import load_weather_for_clc, weather_fields

df_weather = load_weather_for_clc()

print(f"Filas con clima: {len(df_weather):,}")
print(f"Columnas: {df_weather.shape[1]}")
campos_clima = [c for c in weather_fields() if c in df_weather.columns]
print(f"Variables de clima presentes: {len(campos_clima)}/{len(weather_fields())}")
print(f"Filas sin temperature_2m: {df_weather['temperature_2m'].isna().sum()}")

df_weather.head()


In [ ]:
# Celda 4 — Enriquecimiento con cobertura del suelo (CLCPlus Backbone) / Land cover
#
# [ESP] Para cada detección lee el píxel central en el tile CLCPlus que le
#   corresponde (EPSG:4326 → EPSG:3035) y, con el barrio activado en
#   configs/project.yaml (clcplus.neighborhood: anillos 1 y 2), también los 8
#   vecinos → columnas clc_class_N/S/W/E/NW/NE/SW/SE y clc_uniform_surroundings.
#   Al cruzar el borde de un tile el vecino se resuelve en el tile contiguo.
#   Se guarda con nombre de etapa firms_spain_weather_clc.csv: cada CSV de
#   data/processed/enriched/ denota su contenido y alimenta al siguiente paso.
#   (data/raw/clcplus/ es solo lectura.)
#
# [EN] Reads the center CLCPlus pixel per detection (plus the 8 ring 1-2
#   neighbors) and saves the result as the named pipeline stage
#   firms_spain_weather_clc.csv, which feeds the next step.

from wildfire.enrichment.clc_enrichment import enrich_with_clc, save_enriched

df_clc = enrich_with_clc(df_weather, country="Spain")
ruta_clc = save_enriched(df_clc, filename="firms_spain_weather_clc.csv")

print(f"Filas: {len(df_clc):,}")
print(f"Columnas: {df_clc.shape[1]}")
print("Distribución clc_class:")
print(df_clc["clc_class"].value_counts(dropna=False).sort_index().to_string())
print(f"CSV guardado en: {ruta_clc}")


In [ ]:
# Celda 5 — TODO sellado / Sealed TODO
#
# [ESP] TODO (ES/EN) pendiente con el equipo — ¿eliminar filas con
#   clc_class == 1 ("Sealed")? Decisión que afecta al dataset de modelado
#   (firms_spain_final.csv). No descomentar hasta que el equipo decida.
#
# [EN] Pending team decision — drop rows with clc_class == 1 ("Sealed")?
#   Affects the modeling input (firms_spain_final.csv). Do not uncomment
#   until the team decides.
#
# df_clc = df_clc[df_clc["clc_class"] != 1]


In [ ]:
# Celda 6 — Región CCAA + presupuestos / CCAA region + budgets
#
# [ESP] assign_ccaa cruza cada detección con los límites autonómicos
#   (data/raw/ccaa/spain-communities.geojson, fuente decidida por el equipo):
#   1. sjoin "within" en EPSG:3035; los puntos sin polígono que los contenga
#      (costa / frontera PT-FR) usan el respaldo sjoin_nearest con tope
#      ccaa.nearest_max_distance_m = 2000 m; puntos en frontera compartida
#      → se queda con el primer polígono (nunca se duplican filas).
#   2. Los nombres del GeoJSON se traducen al nombre EXACTO del CSV de
#      presupuestos (REGION_NAME_MAP: "Castilla-Leon" → "Castilla y León",
#      "Valencia" → "Comunidad Valenciana"...). Ceuta/Melilla no están
#      en el presupuesto → sus sumas salen NaN.
#   3. add_ccaa_budget_sums → sum_prevention / sum_extinction con ventana
#      móvil de ccaa.years_window=3 años terminada en el año de acq_date
#      (NaN, nunca 0, si la ventana está vacía).
#   4. Guardar firms_spain_final.csv → entrada del modelado.
#
# [EN] Joins each detection with the CCAA boundaries: within-join in
#   EPSG:3035, nearest fallback (≤ ccaa.nearest_max_distance_m) for
#   coastal / border points, border ties deduped keep-first, GeoJSON names
#   translated to the exact budget names, then 3-year rolling budget sums
#   ending at acq_date's year (NaN when empty); saves the modeling input
#   firms_spain_final.csv.
#
from wildfire.data.ccaa import add_ccaa_budget_sums
from wildfire.enrichment.clc_enrichment import save_enriched
from wildfire.geo.regions import assign_ccaa
#
df_final = add_ccaa_budget_sums(assign_ccaa(df_clc))
ruta_final = save_enriched(df_final, filename="firms_spain_final.csv")
#
print(f"Filas: {len(df_final):,}")
print(f"Columnas: {df_final.shape[1]}")
print("Distribución ccaa:")
print(df_final["ccaa"].value_counts(dropna=False).to_string())
print(f"Filas sin ccaa: {int(df_final['ccaa'].isna().sum())}")
print(f"Filas sin sum_prevention: {int(df_final['sum_prevention'].isna().sum())}")
print(f"CSV guardado en: {ruta_final}")
#
df_final.head()
